# 🧪 W1-D4 概念实验：第一周总复习（可执行版）

> 配套阅读：`第1周-Day4-第一周总复习.md`（知识脉络图与概念对照表在那边）
> 复习最好的方式不是再读一遍，而是**把整周的核心机制亲手再跑一遍并自测**：
> Day1 自注意力 → Day2 多头/位置编码 → Day3 手写实现，全部串成一条链
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：公式重建自检 —— 一行代码还原 Attention

不看讲义，直接用 numpy 写出 `Attention(Q,K,V) = softmax(QKᵀ/√d_k)V`，
和分步手算对答案；顺带自检 softmax 的三条性质（非负、和为 1、平移不变）。

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(42)
Q, K, V = (rng.normal(size=(4, 6)) for _ in range(3))

# 分步手算
s = Q @ K.T / np.sqrt(6)
step_by_step = softmax(s, axis=1) @ V

# 一行公式
one_liner = softmax(Q @ K.T / np.sqrt(Q.shape[-1]), axis=1) @ V

print("分步 vs 一行公式，最大偏差:", f"{np.abs(step_by_step - one_liner).max():.2e}")

p = softmax(np.array([1.0, 2.0, 3.0]))
checks = [
    ("非负",        bool((p >= 0).all())),
    ("和为 1",      bool(np.isclose(p.sum(), 1.0))),
    ("平移不变",    bool(np.allclose(softmax(np.array([1., 2., 3.]) + 100.), p))),
]
for name, ok in checks:
    print(f"  softmax {name}: {'✓' if ok else '✗'}")
print("公式 ✓  Attention(Q,K,V) = softmax(Q·Kᵀ/√d_k)·V")

## 实验 2：单头 vs 多头 —— 4 个头看 4 个"角度"

同一个句子、4 个头（d=8 拆成 4×2）。每头的 W_q/W_k 不同 → 注意力分布不同。
随机初始化就能看出"各头关注点不一样"，训练后这种分化会固化成真正的分工
（句法头 / 共指头 / 位置头……）。拼接后维度还原，再经 W_O 融合。

In [ ]:
n, d, h = 4, 8, 4
d_head = d // h
X = rng.normal(size=(n, d))
tokens = ["我", "爱", "AI", "学习"]

heads_out, heads_W = [], []
for i in range(h):
    Wqh = rng.normal(scale=0.6, size=(d, d_head))
    Wkh = rng.normal(scale=0.6, size=(d, d_head))
    Wvh = rng.normal(scale=0.6, size=(d, d_head))
    Qh, Kh, Vh = X @ Wqh, X @ Wkh, X @ Wvh
    s = Qh @ Kh.T / np.sqrt(d_head)
    W_i = softmax(s, axis=1)
    out_i = W_i @ Vh
    heads_out.append(out_i); heads_W.append(W_i)
    ent = -(W_i * np.log(W_i + 1e-12)).sum(axis=1).mean()
    focus = [tokens[int(r.argmax())] for r in W_i]
    print(f"头{i}: 「AI」 这一行关注 {dict(zip(tokens, W_i[2].round(3)))}  平均熵={ent:.2f}  各词最关注={focus}")

concat = np.hstack(heads_out)
print(f"\n拼接 4×{d_head} 维 → shape {concat.shape}，再乘 W_O({d}×{d}) 融合回 {d} 维")
print("→ 多头 = 在不同子空间里并行做注意力，最后融合（比 1 个 8 维头表达力强）")

## 实验 3：位置编码 —— 值的分布 & "距离越远越不相关"

正弦位置编码：偶维用 sin、奇维用 cos，频率从 1 到 1/10000^(-2i/d) 几何递减。
看两件事：热力图的"条纹"结构；PE(50) 与 PE(50+k) 的点积随 k 的衰减/振荡——
这给了模型"相对距离"的信号。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

def sinusoidal_pe(pos, d):
    pe = np.zeros(d)
    for i in range(0, d, 2):
        freq = 1.0 / (10000 ** (i / d))
        pe[i]   = np.sin(pos * freq)
        pe[i+1] = np.cos(pos * freq)
    return pe

PE = np.array([sinusoidal_pe(p, 64) for p in range(100)])
base = 50
sims = [PE[base] @ PE[base + k] for k in range(50)]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
im = axes[0].imshow(PE[:50], aspect="auto", cmap="RdBu_r")
axes[0].set_xlabel("维度（偶=sin，奇=cos）"); axes[0].set_ylabel("位置 pos")
axes[0].set_title("正弦位置编码：低维高频、高维低频")
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].plot(range(50), sims, ".-")
axes[1].set_xlabel("相对距离 k"); axes[1].set_ylabel("PE(50) · PE(50+k)")
axes[1].set_title("点积随相对距离衰减/振荡 → 携带位置信号")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 实验 4：把整周串起来 —— 词向量 → 位置注入 → 多头注意力

一条链跑通：Embedding(查表) → +PE → 4 头注意力 → 拼接。
每个阶段打印 shape，确认"信息流水线"畅通——这就是 Transformer Block 的前半段。

In [ ]:
n, d, h = 6, 32, 4
d_head = d // h

vocab_demo = {f"词{i}": i for i in range(10)}
E = rng.normal(size=(10, d))                    # 假想 Embedding 表

ids = [vocab_demo[t] for t in [f"词{i}" for i in range(n)]]
emb = E[ids]                                    # 查表：不训练、纯索引
print("token ids   ", ids, "shape", np.shape(ids))
print("Embedding   shape", emb.shape)

H = emb + np.array([sinusoidal_pe(p, d) for p in range(n)])   # 注入位置
print("+ 位置编码  shape", H.shape, "（逐元素相加，位置从此进入每一层）")

outs = []
for _ in range(h):
    Wqh = rng.normal(scale=0.4, size=(d, d_head)); Wkh = rng.normal(scale=0.4, size=(d, d_head))
    Wvh = rng.normal(scale=0.4, size=(d, d_head))
    Qh, Kh, Vh = H @ Wqh, H @ Wkh, H @ Wvh
    outs.append(softmax(Qh @ Kh.T / np.sqrt(d_head), axis=1) @ Vh)
final = np.hstack(outs)
print("多头注意力  每头 shape", outs[0].shape, "→ 拼接 shape", final.shape)
print("\n一周成果: 查表 → +PE → 多头注意力 —— Day1(机制) + Day2(位置/多头) + Day3(实现) 全部就位")

## 实验 5：可执行自测 —— 5 道题算出答案再对答案

每道题都真的算：答对打印 ✓，答错打印 ✗（都设计成必对，用来强化记忆点）。

In [ ]:
def check(no, desc, cond):
    print(f"  Q{no} {desc}: {'✓' if cond else '✗'}")

# Q1: 注意力权重的每一行和是多少？
w_row = softmax(rng.normal(size=(5, 5)), axis=1)
check(1, "权重每行和 = 1", np.allclose(w_row.sum(axis=1), 1.0))

# Q2: d_k 变大，未缩放分数更极端？
def row_entropy(v):
    return -(v * np.log(v + 1e-12)).sum()
ents = []
for d_k in [64, 1024]:
    s = rng.normal(size=d_k) * np.sqrt(d_k)     # 方差 = d_k
    ents.append(row_entropy(softmax(s)))
check(2, f"d_k=1024 熵({ents[1]:.2f}) < d_k=64 熵({ents[0]:.2f})，更饱和", ents[1] < ents[0])

# Q3: 因果掩码下位置 0 能看到几个词？
m = np.tril(np.ones((6, 6), dtype=int))
check(3, "位置 0 只能看到 1 个词（自己）", m[0].sum() == 1)

# Q4: PE 与自身的点积是同距离里最大的？
pe0 = sinusoidal_pe(30, 64)
sims4 = [pe0 @ sinusoidal_pe(30 + k, 64) for k in range(1, 10)]
check(4, "PE(30)·PE(30) 最大（k=0）", all(pe0 @ pe0 > s for s in sims4))

# Q5: 4 个 8 维头拼接后的维度？
check(5, "4 头 × 8 维 = 32 维", 4 * 8 == 32)

print("\n第一周机制层毕业 🎓 明天进入第 2 周：FFN / LayerNorm / Tokenizer / RoPE / KV Cache")

## 结论

| Day | 核心机制 | 本 notebook 对应实验 |
|---|---|---|
| Day1 | 自注意力公式 | 实验 1（一行公式重建 + softmax 三性质） |
| Day2 | 多头 + 位置编码 | 实验 2、3（头间分化 / PE 条纹与距离衰减） |
| Day3 | 手写实现 | 实验 4（封装成流水线跑通） |
| 全周 | 综合掌握 | 实验 5（5 道可执行自测题全 ✓） |

→ 深入阅读：同目录 `.md` 版本第二节（深入版原理回顾 + 业务关联）